# Problem 6: BPE 分词 (BPE Tokenization)

在本问题中，你将实现 BPE (Byte Pair Encoding) 分词器，
这是现代大语言模型常用的分词算法。

**主要参考**: [Sennrich et al., 2016](https://arxiv.org/abs/1508.07909)

## 6.1 训练 BPE 分词器 (Training a BPE Tokenizer)

**目标**: 实现 `run_train_bpe` 函数。

BPE 是一种数据驱动的子词分词算法，通过迭代合并最频繁的字节对来构建词汇表。

---

### BPE 算法流程

**初始化**:
1. 将训练文本编码为字节序列（使用 UTF-8 编码）
2. 初始词汇表包含所有单个字节（0-255，共 256 个）
3. 添加特殊 token（如 `<pad>`, `<eos>`, 等）到词汇表
4. 为每个 token 分配唯一 ID

**迭代合并**:

重复以下步骤，直到词汇表大小达到 `vocab_size`：

1. **统计字节对频率**:
   - 遍历整个训练语料
   - 统计每个相邻字节对 `(byte1, byte2)` 出现的次数

2. **选择最频繁的字节对**:
   - 找到出现次数最多的字节对 `(best_pair)`

3. **合并**:
   - 在词汇表中添加新的 token：`best_pair` 的合并结果
   - 记录合并操作：`merges.append(best_pair)`
   - 在语料中，所有出现这个字节对的地方合并为单个 token

4. **更新词汇表**:
   - 新 token 的 ID = 当前词汇表大小

---

### 算法示例

假设训练语料是: `"hug hug pug hug pug"`

**初始状态**:
- 字节序列（简化为字符）: `['h', 'u', 'g', ' ', 'h', 'u', 'g', ' ', 'p', 'u', 'g', ...]`
- 初始词汇表: `{0: b'h', 1: b'u', 2: b'g', 3: b' ', 4: b'p', ...}`

**第 1 次迭代**:
- 统计字节对频率:
  - `('h', 'u')`: 3 次
  - `('u', 'g')`: 3 次
  - `('g', ' ')`: 2 次
  - ...
- 最频繁对: `('h', 'u')` 出现 3 次
- 合并: 新 token `b'hu'` 加入词汇表
- 更新语料: `['hu', 'g', ' ', 'hu', 'g', ' ', 'p', 'u', 'g', ...]`

**第 2 次迭代**:
- 统计字节对频率:
  - `('hu', 'g')`: 3 次 (现在是 'hu' 和 'g' 这个对)
  - `('g', ' ')`: 2 次
  - ...
- 最频繁对: `('hu', 'g')` 出现 3 次
- 合并: 新 token `bhug'` 加入词汇表
- 更新语料: `['hug', ' ', 'hug', ' ', 'p', 'u', 'g', ...]`

**继续迭代**:
- 直到词汇表大小达到 `vocab_size`

---

### 数据结构

**词汇表** (`vocab`):
```python
vocab = {
    0: b'<pad>',       # 特殊 token
    1: b'<eos>',       # 特殊 token
    2: b'h',          # 单字节
    3: b'u',
    4: b'g',
    5: b' ',
    6: b'hu',         # 合并后的 token
    7: b'hug',        # 合并后的 token
    ...
}
# 映射: token_id -> token_bytes
```

**合并列表** (`merges`):
```python
merges = [
    (b'h', b'u'),     # 第 1 次合并: h + u -> hu
    (b'hu', b'g'),    # 第 2 次合并: hu + g -> hug
    ...
]
# 列表顺序很重要！重建时必须按相同顺序应用
```

---

### 实现要点

**1. 文件读取**:
- 从 `input_path` 读取文本文件
- 使用 UTF-8 编码

**2. 字节编码**:
- 使用 `text.encode('utf-8')` 将字符串转换为字节序列
- 例如: `"hello".encode('utf-8')` -> `b'hello'`

**3. 特殊 Token**:
- `special_tokens` 是字符串列表: `['<pad>', '<eos>']`
- 需要先添加到词汇表（优先级最高）
- 编码为 UTF-8 字节: `<pad>` -> `b'<pad>'`

**4. 统计字节对**:
- 遍历所有文本的字节序列
- 对每个相邻的 `(bytes[i], bytes[i+1])` 对计数
- 使用字典或 Counter 统计频率

**5. 合并操作**:
- 选择频率最高的字节对
- 如果有多个对具有相同最高频率，选择**字典序最小**的
- 更新语料库，将所有该字节对替换为新的合并 token

**6. 返回值**:
```python
return vocab, merges
# vocab: Dict[int, bytes] - token_id -> token_bytes
# merges: List[Tuple[bytes, bytes]] - 按顺序的合并操作
```

---

### 对应函数

`tests/adapters.py` 中的 `run_train_bpe(input_path, vocab_size, special_tokens, **kwargs)`

**参数**:
- `input_path`: 训练文本文件路径
- `vocab_size`: 最终词汇表大小（包括特殊 token）
- `special_tokens`: 特殊 token 字符串列表

**返回**:
- `vocab`: `Dict[int, bytes]` - token ID 到 token 字节的映射
- `merges`: `List[Tuple[bytes, bytes]]` - 按顺序的合并操作列表

---

### 提示

1. **效率优化**: 统计字节对频率时，不要每次都遍历整个语料库
2. **数据结构**: 考虑使用列表的列表来表示语料，方便合并操作
3. **特殊情况**: 处理只有单个字节的情况，空文件等边界情况

## 6.2 加载 BPE 分词器 (Loading a BPE Tokenizer)

**目标**: 实现 `get_tokenizer` 函数。

这个函数返回一个可以使用训练好的 BPE 模型进行编码和解码的分词器对象。

---

### 分词器的功能

一个 BPE 分词器需要支持以下操作：

**1. 编码 (Encoding)**:
   - 将文本字符串转换为 token IDs 列表
   - 步骤: 文本 -> 字节 -> 应用 merges -> token IDs

**2. 解码 (Decoding)**:
   - 将 token IDs 列表转换回文本字符串
   - 步骤: token IDs -> token bytes -> 拼接 -> 文本

---

### 编码流程

给定输入文本 `"hello world"`:

1. **转换为字节**:
   ```python
   text = "hello world"
   bytes_list = list(text.encode('utf-8'))  # [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]
   ```

2. **应用合并规则** (按 merges 列表的顺序):
   ```python
   # 初始: [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]
   # 应用 merges[0]: (b'h', b'e') -> b'he'
   # 结果: [b'he', 108, 108, 111, 32, 119, 111, 114, 108, 100]
   # 继续应用更多合并规则...
   ```

3. **转换为 token IDs**:
   - 在 vocab 中查找每个 token 对应的 ID
   - 返回 token IDs 列表

---

### 解码流程

给定 token IDs `[23, 45, 67]`:

1. **查找 token bytes**:
   ```python
   tokens = [vocab[23], vocab[45], vocab[67]]  # [b'h', b'el', b'lo']
   ```

2. **拼接字节**:
   ```python
   text_bytes = b''.join(tokens)  # b'hello'
   ```

3. **解码为文本**:
   ```python
   text = text_bytes.decode('utf-8')  # 'hello'
   ```

---

### 特殊 Token 处理

特殊 token（如 `<pad>`, `<eos>`）有以下特点：

1. **永不分割**: 特殊 token 作为单个 token，不会参与 BPE 合并
2. **优先匹配**: 编码时，如果文本中包含特殊 token，应该直接匹配
3. **独立存在**: 特殊 token 在词汇表中独立，不能通过合并得到

例如:
- `"Hello <eos> world"` -> `[token_ids_for('Hello'), eos_id, token_ids_for('world')]`
- `"<pad>"` -> `[pad_id]` (单个 token)

---

### 实现要求

**返回对象**:
- 可以是任何实现了编码和解码功能的对象
- 通常是一个类实例，有以下方法：
  - `encode(text: str) -> List[int]`
  - `decode(token_ids: List[int]) -> str`

**使用 `vocab` 和 `merges`**:
- `vocab`: `Dict[int, bytes]` - 从 token ID 到 token bytes 的映射
- `merges`: `List[Tuple[bytes, bytes]]` - 训练时的合并顺序

**提示**:
- 可以考虑实现一个 `BPETokenizer` 类
- 为了快速查找，可能需要建立 `bytes -> int` 的反向映射
- 编码时使用贪婪匹配：优先匹配长 token

---

### 对应函数

`tests/adapters.py` 中的 `get_tokenizer(vocab, merges, special_tokens)`

**参数**:
- `vocab`: `Dict[int, bytes]` - token ID 到 token bytes 的映射
- `merges`: `List[Tuple[bytes, bytes]]` - BPE 合并规则（按顺序）
- `special_tokens`: `List[str]` - 特殊 token 列表（可选）

**返回**:
- 一个分词器对象，支持 `encode()` 和 `decode()` 操作